In [ ]:
'''
To improve from prev version of makemore:

Initialization is off - the loss at initialization should be close to expected loss
- Loss at initialization should be that of equal probability (for this case)

DEAD NEURONS:
- activations (h): pre-activations have extreme (pos/neg) values making their tanh close to 1/-1
- remember: tanh squashes values down to be in between -1 and 1
- derivative of tanh (backward pass) = (1 - tanh^2)*grad
- tanh values close to 1/-1 cause the gradient to go to 0
- Bad because gradient could be high but because of chain rule for backward pass we do gradient*deriv of tanh so backward pass grad would be 0
- means changing weights/biases don't impact loss because in flat region of tanh

Batch Normalization
- Initiates to Gaussian: every neuron's activation will be gaussian across the number of inputs at initiation only
- backpropagate through bnbias and bngain to enable network to optimize and not be set to gaussian in each iteration
- Customary to append a batch normalization layer after linear or convolutional layer to control scale of activations at every layer in net
- Don't need biases for normalization layers -> will be encompassed in bnbias
- Karpathy's summary:
----- Use bn to control statistics of activations in neural net
----- Sprinkle bn across neural net, typically after layers with multiplication (linear or convolutional layer)
----- parameters: bngain and bnbias -> trained in back propagation
----- buffers: running mean and stdev -> trained using running mean update
----- centers batch to be unit gaussian -> offsets and scales based on training
- Likely to cause bugs in code due to coupling of inputs during the forward pass
- Can alternatively use group or layer normalization

Resnet50
- Convolutional layers used for images because they have spacial structure
- linear multiplication and bias offset are done in patches instead of on full input
- uses relu as nonlinearity -> better for very deep networks

Pytorch
- nn.Linear(num, num)
----- could also initiate linear layers in pytorch -> only differences are ignores gain and uses uniform distribution by default
----- if you have a roughly gaussian input, this will give you a roughly gaussian output
- batchnorm1D -> batch normalization layer
----- momentum is used for mean and stdev tracking (we used 0.001)
----- (learnable) affine parameters: gain and bias
----- track_running_stats = track the running mean and stdev like we did or you can also calculate it after optimization as a phase 2
'''

# Original:
train 2.035
val 2.168

# Fixed softmax confidently wrong (loss at initialization)
train 2.07
val 2.13

# Fixed tanh layer too saturated at initialization
train 2.036
val 2.103

# Batch normalization - more useful for more complex, more layered networks
train 2.0517263412475586
val 2.106353282928467

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inLine

words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [2]:
len(words)

32033

In [3]:
# build the vocabulary of characters and mappings
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [4]:
# build the data set
block_size = 3 # context length: how many chars are considered when predicting the next char

def build_dataset(words):
    X, Y = [],[]

    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context) # stores the context for each char
            Y.append(ix) # stores the char
            context = context[1:] + [ix] # crop and append the new char to context

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1]) # Training set (80%)
Xdev, Ydev = build_dataset(words[n1:n2]) # Dev set (10%)
Xte, Yte = build_dataset(words[n2:]) # Test set (10%)

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [5]:
# MLP

n_embd = 10 # dimensionality of the character embedding vectors
n_hidden = 200 # number of neurons in hidden layer of MLP

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocab_size, n_embd), generator = g) # embedding vector: initiated as uniform gaussian
W1 = torch.randn((n_embd * block_size, n_hidden), generator = g) * (5/3)/((n_embd * block_size)**0.5) # for initialization so hpreact is close to 0
b1 = torch.randn(n_hidden, generator = g) * 0.01 # for initialization so hpreact is close to 0
W2 = torch.randn((n_hidden, vocab_size), generator = g) * 0.01 # scale down for initialization because we want all logits to be equal
b2 = torch.randn(vocab_size, generator = g) * 0 # for initialization because we want all logits to be equal

bngain = torch.ones((1, n_hidden)) # batch normalization gain
bnbias = torch.zeros((1, n_hidden)) # batch normalization bias
bnmean_running = torch.zeros((1, n_hidden)) # store batch means during training
bnstd_running = torch.ones((1, n_hidden)) # store batch stdevs during training

parameters = [C, W1, b1, W2, b2]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
    p.requires_grad = True


'''
C = embedding vector
- dimensions: 27, 10 -> 27 char inputs each with an embedding of 10 values

W1 = inputs
- dimensions: 30, 200 -> context is 3 chars * 10 embeddings per car, connected to 200 neurons in hidden layer

W2 = inputs from the hidden layer to the output layer
- dimensions: 200, 27 -> 200 neurons feed into 27 char options
'''

11897


'\nC = embedding vector\n- dimensions: 27, 10 -> 27 char inputs each with an embedding of 10 values\n\nW1 = inputs\n- dimensions: 30, 200 -> context is 3 chars * 10 embeddings per car, connected to 200 neurons in hidden layer\n\nW2 = inputs from the hidden layer to the output layer\n- dimensions: 200, 27 -> 200 neurons feed into 27 char options\n'

In [ ]:
# KAIMING NORMAL on pytorch explained

'''
**see paper: https://arxiv.org/abs/1502.01852
To avoid dead neurons:
- Divide initialized weights by sqrt(dimensions of x/inputs)
- For different activation functions this will be different -> for some will need to scale with a "gain" to preserve the gaussian
- Proper initialization of the forward pass yields an approximately initiated backward pass as well

More recent innovations have made initialization less important
- residual connections
- use of normalization layers
- better optimizers (more complex than gradient descent)
'''

x = torch.randn(1000,10) # 1000 10-dimensional examples
w = torch.randn(10,200) / 10**0.5 # scale down such that stdev(y) = 1 to preserve gaussian
y = x @ w
print(x.mean(), x.std())
print(y.mean(), y.std())
plt.figure(figsize=(20,5))
plt.subplot(121)
plt.hist(x.view(-1).tolist(), 50, density=True);
plt.subplot(122)
plt.hist(y.view(-1).tolist(), 50, density=True);

In [ ]:
'''
# Optimization - pre batch normalization

max_steps = 200000
batch_size = 32
lossi = []

for i in range(max_steps):

    # mini batch construct
    ix = torch.randint(0, Xtr.shape[0], (batch_size,),generator = g) # randomly selects a mini batch size
    Xb, Yb = Xtr[ix], Ytr[ix] # mini batch

    # forward pass
    emb = C[Xb] # embed the chars into vectors
    embcat = emb.view(emb.shape[0], -1) # concatenates the vectors for multiplication
    hpreact = embcat @ W1 + b1 # hidden layer pre-activation
    h = torch.tanh(hpreact) # hidden layer (activated)
    logits = h @ W2 + b2 # output layer
    loss = F.cross_entropy(logits, Yb) # loss function

    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # update
    lr = 0.1 if i < 100000 else 0.01 # step learning rate decay
    for p in parameters:
        p.data += -lr * p.grad

    # track stats
    if i % 10000 == 0: # print every 10,000 iterations
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
    lossi.append(loss.log10().item())
'''

In [43]:
# Batch Normalization: Initiates to Gaussian: every neuron's activation will be gaussian across the number of inputs at initiation only

# Optimization

max_steps = 200000
batch_size = 32
lossi = []

for i in range(max_steps):

    # mini batch construct
    # activations will differ (jitter) between batches because they depend on what other examples are in the batch (which is random)
    # works as a regularizer -> harder to overfit the network
    ix = torch.randint(0, Xtr.shape[0], (batch_size,),generator = g) # randomly selects a mini batch size
    Xb, Yb = Xtr[ix], Ytr[ix] # mini batch

    # forward pass
    emb = C[Xb] # embed the chars into vectors
    embcat = emb.view(emb.shape[0], -1) # concatenates the vectors for multiplication
    hpreact = embcat @ W1 + b1 # hidden layer pre-activation - before batch normalization
    bnmeani = hpreact.mean(0, keepdim=True)
    bnstdi = hpreact.std(0, keepdim=True)
    hpreact = bngain * (hpreact - bnmeani) / bnstdi + bnbias # Batch normalization, add very small number to prevent division by 0

    with torch.no_grad():
        bnmean_running = 0.999 * bnmean_running + 0.001 * bnmeani
        bnstd_running = 0.999 * bnstd_running + 0.001 * bnstdi
    
    h = torch.tanh(hpreact) # hidden layer (activated)
    logits = h @ W2 + b2 # output layer
    loss = F.cross_entropy(logits, Yb) # loss function

    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # update
    lr = 0.1 if i < 100000 else 0.01 # step learning rate decay
    for p in parameters:
        p.data += -lr * p.grad

    # track stats
    if i % 10000 == 0: # print every 10,000 iterations
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
    lossi.append(loss.log10().item())

      0/ 200000: 3.3054
  10000/ 200000: 1.8221
  20000/ 200000: 1.8818
  30000/ 200000: 2.5004
  40000/ 200000: 2.4503
  50000/ 200000: 1.9280
  60000/ 200000: 2.4076
  70000/ 200000: 2.2211
  80000/ 200000: 1.7427
  90000/ 200000: 2.3965
 100000/ 200000: 2.2330
 110000/ 200000: 2.1697
 120000/ 200000: 1.9552
 130000/ 200000: 1.8198
 140000/ 200000: 2.2674
 150000/ 200000: 2.1655
 160000/ 200000: 1.9447
 170000/ 200000: 2.1909
 180000/ 200000: 2.4979
 190000/ 200000: 1.8792


In [46]:
# calibrate the batch norm at the end of training
# without this, the network input is batches, so you can't feed in a single example for an output

with torch.no_grad(): # for efficiency
    # pass the training set through
    emb = C[Xtr]
    embcat = emb.view(emb.shape[0], -1)
    hpreact = embcat @ W1 + b1
    # measure the mean/stdev over entire training set
    bnmean = hpreact.mean(0, keepdim=True)
    bnstd = hpreact.std(0, keepdim=True)

bnstd

tensor([[2.6873, 2.0130, 2.3784, 2.3864, 2.2754, 2.3267, 2.0830, 2.3597, 2.4359,
         2.1422, 2.6191, 2.2583, 2.0898, 2.2540, 2.1752, 2.6367, 2.4059, 2.0607,
         2.2683, 2.6094, 2.1938, 2.2892, 2.1069, 2.1338, 2.2381, 2.4045, 2.2547,
         2.4206, 2.5041, 2.4701, 2.1134, 2.1674, 2.2907, 2.1479, 2.1493, 2.1424,
         2.3518, 2.5826, 2.9461, 1.9941, 2.2144, 2.2128, 2.3979, 2.1095, 2.4929,
         2.4991, 2.2116, 2.7722, 2.2163, 2.5140, 2.0257, 2.0421, 1.8150, 1.9511,
         2.2216, 2.3494, 1.9866, 2.2530, 2.8308, 2.0380, 2.4037, 2.2886, 1.9700,
         2.4007, 2.3068, 2.1213, 2.1070, 2.5294, 2.1616, 2.2297, 2.1265, 2.0311,
         2.0180, 2.2132, 1.8418, 1.9172, 2.5027, 2.5042, 2.0581, 2.1724, 2.4924,
         2.0763, 2.2405, 2.4079, 2.5486, 2.2828, 2.1720, 2.3633, 2.5810, 2.6950,
         1.9640, 1.9333, 2.7432, 2.1086, 2.2224, 2.0266, 2.1498, 2.1406, 2.1412,
         2.2189, 1.9388, 2.1241, 2.1066, 2.0990, 2.0600, 2.0600, 2.1579, 2.2048,
         2.1295, 2.3539, 2.0

In [47]:
bnstd_running

tensor([[2.6763, 1.9888, 2.3629, 2.3691, 2.2524, 2.3119, 2.0690, 2.3283, 2.4127,
         2.1226, 2.6134, 2.2380, 2.0670, 2.2369, 2.1507, 2.6000, 2.3876, 2.0396,
         2.2384, 2.6010, 2.1710, 2.2813, 2.0891, 2.1030, 2.2143, 2.3970, 2.2313,
         2.3923, 2.4829, 2.4491, 2.0876, 2.1510, 2.2759, 2.1141, 2.1285, 2.1312,
         2.3384, 2.5774, 2.9441, 1.9719, 2.1841, 2.1879, 2.3738, 2.0879, 2.4713,
         2.4760, 2.1905, 2.7352, 2.2091, 2.4965, 2.0072, 2.0222, 1.7890, 1.9286,
         2.2012, 2.3308, 1.9686, 2.2287, 2.8126, 2.0174, 2.3888, 2.2630, 1.9543,
         2.3592, 2.2809, 2.0946, 2.0886, 2.4960, 2.1478, 2.2086, 2.1023, 2.0100,
         1.9947, 2.1868, 1.8223, 1.8944, 2.4725, 2.4878, 2.0330, 2.1440, 2.4782,
         2.0564, 2.2195, 2.3804, 2.5172, 2.2731, 2.1434, 2.3389, 2.5635, 2.6547,
         1.9436, 1.9108, 2.7419, 2.0963, 2.2030, 2.0045, 2.1209, 2.1157, 2.1128,
         2.1949, 1.9250, 2.0935, 2.0952, 2.0737, 2.0345, 2.0381, 2.1325, 2.1833,
         2.1096, 2.3341, 2.0

In [48]:
@torch.no_grad() # this decorator disables gradient tracking for efficiency (not storing all info needed to later do a backward pass)

def split_loss(split): # calculates the loss of each split
    x, y = {
        'train': (Xtr, Ytr),
        'val': (Xdev, Ydev),
        'test': (Xte, Yte),
    }[split]
    emb = C[x] # N, block_size, n_embd
    embcat = emb.view(emb.shape[0], -1) # concatenate into N, block_size, n_embd
    hpreact = embcat @ W1 + b1 # hidden layer pre-activation - before batch normalization
    hpreact = bngain * (hpreact - bnmean_running) / bnstd_running + bnbias # Batch normalization
    h = torch.tanh(hpreact) # (N, n_hidden)
    logits = h @ W2 + b2 # (N, vocab_size)
    loss = F.cross_entropy(logits, y)
    print(split, loss.item())

split_loss('train')
split_loss('val')

train 2.0761094093322754
val 2.1306095123291016


In [ ]:
# sample from model
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):
    out = []
    context = [0] * block_size # initialize with all "..."
    while True:
        # forward pass the neural net
        emb = C[torch.tensor([context])] # (1, block_size, n_embd)
        h = torch.tanh(emb.view(1, -1) @ W1 + b1)
        logits = h @ W2 + b2
        probs = F.softmax(logits, dim=1)

        # sample from distribution
        ix = torch.multinomial(probs, num_samples=1, generator = g).item()

        # shift the context and track samples
        context = context[1:] + [ix]
        out.append(ix)

        # if we sample '.' then break
        if ix == 0:
            break
    print(''.join(itos[i] for i in out)) # decode and print the generated word